### EMBEDDING AND VECTOR DB

In [2]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

In [6]:
CHUNKS_PATH     = "../data/processed/chunks.json"
CHROMA_PATH     = "../data/processed/chroma_db"
COLLECTION_NAME = "elte_ik"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

In [7]:
class EmbeddingPipeline:
    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model = SentenceTransformer(model_name)
        print(f"Loaded model: {model_name}")

    def encode(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=True)

In [ ]:
with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks")

pipeline = EmbeddingPipeline()
texts = [c["content"] for c in chunks]
embeddings = pipeline.encode(texts)
print(f"Embeddings shape: {embeddings.shape}") 

Loaded 11 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\boroh\ELTE\Thesis\elte_chat\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\boroh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (11, 384)


In [9]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

collection.upsert(
    ids=[str(c["metadata"]["chunk_id"]) for c in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[c["metadata"] for c in chunks]
)
print(f"Upserted {collection.count()} documents into '{COLLECTION_NAME}'")

Upserted 11 documents into 'elte_ik'


In [10]:
query = "What are the prerequisites for enrollment?"
query_emb = pipeline.encode([query]).tolist()

results = collection.query(query_embeddings=query_emb, n_results=3)
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n--- Result {i+1} (chunk {meta['chunk_id']}, {meta['file_name']}) ---")
    print(doc[:300])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Result 1 (chunk 6, The prerequisites.pdf) ---
The strong prerequisite
▪ Strong prerequisites are prerequisites without which the the follow-
up subject cannot be taken in the next semester, and the follow-up 
subject can only be registered for after the prerequisite subject has 
been fulfilled. The strong prerequisite must be completed before 


--- Result 2 (chunk 5, The prerequisites.pdf) ---
The prerequisites
What does prerequisite mean?
They are requirements you must meet before taking certain subjects (follow-up subjects). There are two types of 
prerequisites: strong prerequisites and weak prerequisites.
These will be discussed in the following.
Follow-up 
subject
Follow-up 
subject


--- Result 3 (chunk 10, The prerequisites.pdf) ---
Markings of the prerequisites in the curriculum
▪ Strong prerequisites have no special annotation,
▪ Weak prerequisites are marked as ‘weak’.


In [11]:
import pandas as pd

# Fetch all stored documents
all_data = collection.get(include=["documents", "metadatas", "embeddings"])

df = pd.DataFrame({
    "chunk_id": [m["chunk_id"] for m in all_data["metadatas"]],
    "file_name": [m["file_name"] for m in all_data["metadatas"]],
    "file_type": [m["file_type"] for m in all_data["metadatas"]],
    "content_preview": [d[:80] + "..." for d in all_data["documents"]],
    "embedding_dim": [len(e) for e in all_data["embeddings"]],
})
print(df.to_string(index=False))

 chunk_id                        file_name file_type                                                                        content_preview  embedding_dim
        0 ELTE Faculty of Informatics.html      html ELTE Faculty of Informatics\nSkip to main content\nELTE Faculty of Informatics\nBSc...            384
        1 ELTE Faculty of Informatics.html      html  23.02.2026.\nBecome an EU Careers Student Ambassador (2026–2027)\nELTE, in coopera...            384
        2 ELTE Faculty of Informatics.html      html  Student Erasmus Applications for the 2026/27 Academic Year.\n03.02.2026.\nStudent ...            384
        3 ELTE Faculty of Informatics.html      html  Educational Materials Building Resilience Against Disinformation\nMore News\nEvent...            384
        4 ELTE Faculty of Informatics.html      html   The Wonders of Machine Perception: Sensors and What Lies Behind Them\nThe Signals...            384
        5            The prerequisites.pdf       pdf  The prerequisite